In [1]:
import os
import argparse
import torch
import torch.utils.data
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torchvision import datasets, transforms

In [2]:

# import pdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


In [36]:
from data import data_reader, data_loader
from utils import helpers

In [37]:
class CAE(nn.Module):
	def __init__(self):
		super(CAE, self).__init__()

		self.fc1 = nn.Linear(7, 64, bias = False) # Encoder
		self.fc2 = nn.Linear(64, 7, bias = False) # Decoder

		self.relu = nn.ReLU()
		self.sigmoid = nn.Sigmoid()


	def encoder(self, x):
		h1 = self.relu(self.fc1(x.view(-1, 7)))
		return h1

	def decoder(self,z):
		h2 = self.sigmoid(self.fc2(z))
		return h2

	def forward(self, x):
            h1 = self.encoder(x)
            h2 = self.decoder(h1)
            return h1, h2

        # Writing data in a grid to check the quality and progress
	# def samples_write(self, x, epoch):
	# 	_, samples = self.forward(x)
	# 	#pdb.set_trace()
	# 	samples = samples.data.cpu().numpy()[:16]
	# 	fig = plt.figure(figsize=(4, 4))
	# 	gs = gridspec.GridSpec(4, 4)
	# 	gs.update(wspace=0.05, hspace=0.05)
	# 	for i, sample in enumerate(samples):
	# 		ax = plt.subplot(gs[i])
	# 		plt.axis('off')
	# 		ax.set_xticklabels([])
	# 		ax.set_yticklabels([])
	# 		ax.set_aspect('equal')
	# 		plt.imshow(sample.reshape(28, 28), cmap='Greys_r')
	# 	if not os.path.exists('out/'):
	# 		os.makedirs('out/')
	# 	plt.savefig('out/{}.png'.format(str(epoch).zfill(3)), bbox_inches='tight')
	# 	#self.c += 1
	# 	plt.close(fig)

In [38]:
config = helpers.Config()
cfg = config.from_json("data")
data_read = data_reader.DataReader()
X, y = data_read.load_standardize_data('wind_forecast_2009')
data_load = data_loader.DataModelLoader(X, y)
train_loader = data_load.all_data_loader()

In [39]:
mse_loss = nn.BCELoss(reduction='sum')

In [56]:
def loss_function(W, x, recons_x, h, lam):
    mse = mse_loss(recons_x, x)
    # Since: W is shape of N_hidden x N. So, we do not need to transpose it as
    # opposed to #1
    dh = h * (1 - h) # Hadamard product produces size N_batch x N_hidden
    # Sum through the input dimension to improve efficiency, as suggested in #1
    w_sum = torch.sum(Variable(W)**2, dim=1)
    # unsqueeze to avoid issues with torch.mv
    w_sum = w_sum.unsqueeze(1) # shape N_hidden x 1
    contractive_loss = torch.sum(torch.mm(dh**2, w_sum), 0)
    return mse + contractive_loss.mul_(lam)

In [57]:

lam = 1e-4

In [62]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ", device)

device:  cuda


In [63]:
model = CAE().to(device)
optimizer = optim.Adam(model.parameters(), lr = 0.0001)

In [66]:

def train(epoch):
    model.train()
    train_loss = 0

    for idx, (data1, label) in enumerate(train_loader):

        data = torch.cat((data1, label), dim=1)
        data = data.to(device)

        optimizer.zero_grad()

        hidden_representation, recons_x = model(data)

        # Get the weights
        # model.state_dict().keys()
        # change the key by seeing the keys manually.
        # (In future I will try to make it automatic)
        W = model.state_dict()['fc1.weight']
        loss = loss_function(W, data.view(-1, 7), recons_x,
                             hidden_representation, lam)

        loss.backward()
        train_loss += loss.data[0]
        optimizer.step()

        if idx % 5 == 0:
            print('Train epoch: {} [{}/{}({:.0f}%)]\t Loss: {:.6f}'.format(
                  epoch, idx*len(data), len(train_loader),
                  100*idx/len(train_loader),
                  loss.data[0]/len(data)))


    print('====> Epoch: {} Average loss: {:.4f}'.format(
         epoch, train_loss / len(train_loader)))
    # model.samples_write(data,epoch)


In [67]:
for epoch in range(50):
    train(epoch)

Train epoch: 0 [0/966(0%)]	 Loss: -19280.087891
Train epoch: 0 [640/966(1%)]	 Loss: -18155.218750
Train epoch: 0 [1280/966(1%)]	 Loss: -17875.789062
Train epoch: 0 [1920/966(2%)]	 Loss: -16707.281250
Train epoch: 0 [2560/966(2%)]	 Loss: -18728.812500
Train epoch: 0 [3200/966(3%)]	 Loss: -18511.445312
Train epoch: 0 [3840/966(3%)]	 Loss: -18112.501953
Train epoch: 0 [4480/966(4%)]	 Loss: -20010.718750
Train epoch: 0 [5120/966(4%)]	 Loss: -17727.230469
Train epoch: 0 [5760/966(5%)]	 Loss: -18993.576172
Train epoch: 0 [6400/966(5%)]	 Loss: -18334.664062
Train epoch: 0 [7040/966(6%)]	 Loss: -16923.587891
Train epoch: 0 [7680/966(6%)]	 Loss: -18062.419922
Train epoch: 0 [8320/966(7%)]	 Loss: -18933.324219
Train epoch: 0 [8960/966(7%)]	 Loss: -19583.478516
Train epoch: 0 [9600/966(8%)]	 Loss: -18218.796875
Train epoch: 0 [10240/966(8%)]	 Loss: -19057.101562
Train epoch: 0 [10880/966(9%)]	 Loss: -20452.359375
Train epoch: 0 [11520/966(9%)]	 Loss: -18651.625000
Train epoch: 0 [12160/966(10%)]	

KeyboardInterrupt: 

In [ ]:
6580